In [1]:
# 1
import requests as rq
from bs4 import BeautifulSoup

url = "https://edition.cnn.com/business/tech"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/118.0.0.0 Safari/537.36"
    )
}

try:
    res = rq.get(url, headers=headers, timeout=10)
    res.raise_for_status()
except rq.exceptions.HTTPError as e:
    print("❌ HTTP Error:", e)
    res = None
except rq.exceptions.ConnectionError:
    print("❌ Connection Error: Check your internet!")
    res = None
except rq.exceptions.Timeout:
    print("❌ Timeout Error!")
    res = None
except Exception as e:
    print("❌ Unexpected Error:", e)
    res = None
else:
    print("Page fetched successfully!✅ ")

if res is None:
    raise SystemExit("Request failed, cannot continue.")

res.encoding = "utf-8"
soup = BeautifulSoup(res.text, "lxml")

titles = []
summaries = []

for block in soup.find_all("div", class_="container__headline"):
    title_tag = block.find("span", class_="container__headline-text")
    if title_tag:
        titles.append(title_tag.get_text())

for p in soup.find_all("p"):
    summaries.append(p.get_text())


print("number of titles:", len(titles))
print("number of summaries:", len(summaries))

if titles:
    print("Sample title:", titles[0])
if summaries:
    print("Sample summary:", summaries[0])

raw_text = " . ".join(titles + summaries)
print("Raw text (first 400 chars):")
print(raw_text[:400])

Page fetched successfully!✅ 
number of titles: 27
number of summaries: 7
Sample title: The hottest new AI company is…Google?
Sample summary: Markets 



Raw text (first 400 chars):
The hottest new AI company is…Google? . Wall Street is relying on the Supreme Court to protect the Fed. Is that wishful thinking? . How America’s biggest retailers are preparing for an unpredictable holiday season . The hottest new AI company is…Google? . What bubble? The analysts and investors making the bull case for AI . Lawsuit alleges social media giants buried their own research on teen ment


In [2]:
# 2
import re
def clean_text(text: str) -> str:
    """Clean the text (remove digits, punctuation, extra spaces, ...) """
    text = text.lower()
    text = re.sub(r"\d+", " ", text)  #remove digits
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)  # collapse spaces/newlines/tabs to 1 space
    text = text.strip()
    return text

cleaned_text = clean_text(raw_text)

print("Cleaned text (first 400 chars):")
print(cleaned_text[:400])

Cleaned text (first 400 chars):
the hottest new ai company is google wall street is relying on the supreme court to protect the fed is that wishful thinking how america s biggest retailers are preparing for an unpredictable holiday season the hottest new ai company is google what bubble the analysts and investors making the bull case for ai lawsuit alleges social media giants buried their own research on teen mental health harms


In [3]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 93.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
# 3
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(cleaned_text)

tokens = [token.text for token in doc if not token.is_space]

print("total tokens:", len(tokens))
print("Sample 20 tokens:", tokens[:20])

total tokens: 519
Sample 20 tokens: ['the', 'hottest', 'new', 'ai', 'company', 'is', 'google', 'wall', 'street', 'is', 'relying', 'on', 'the', 'supreme', 'court', 'to', 'protect', 'the', 'fed', 'is']


In [5]:
# 4 -  Stopwords
content_tokens = [
    token for token in doc
    if not token.is_space and not token.is_punct and not token.is_stop
]

print("tokens without stopwords:", len(content_tokens))
print("Sample 20 content tokens:", [t.text for t in content_tokens[:20]])

tokens without stopwords: 351
Sample 20 content tokens: ['hottest', 'new', 'ai', 'company', 'google', 'wall', 'street', 'relying', 'supreme', 'court', 'protect', 'fed', 'wishful', 'thinking', 'america', 's', 'biggest', 'retailers', 'preparing', 'unpredictable']


In [6]:
# 5
print("POS samples (first 40 tokens):")
for token in content_tokens[:40]:
    print(f"{token.text:15}  POS={token.pos_:10}  TAG={token.tag_}")


POS samples (first 40 tokens):
hottest          POS=ADJ         TAG=JJS
new              POS=ADJ         TAG=JJ
ai               POS=PROPN       TAG=NNP
company          POS=NOUN        TAG=NN
google           POS=PROPN       TAG=NNP
wall             POS=PROPN       TAG=NNP
street           POS=PROPN       TAG=NNP
relying          POS=VERB        TAG=VBG
supreme          POS=PROPN       TAG=NNP
court            POS=PROPN       TAG=NNP
protect          POS=VERB        TAG=VB
fed              POS=PROPN       TAG=NNP
wishful          POS=ADJ         TAG=JJ
thinking         POS=NOUN        TAG=NN
america          POS=PROPN       TAG=NNP
s                POS=PART        TAG=POS
biggest          POS=ADJ         TAG=JJS
retailers        POS=NOUN        TAG=NNS
preparing        POS=VERB        TAG=VBG
unpredictable    POS=ADJ         TAG=JJ
holiday          POS=NOUN        TAG=NN
season           POS=NOUN        TAG=NN
hottest          POS=ADJ         TAG=JJS
new              POS=ADJ         T

In [7]:
# 6
print(doc.ents)
entities = [(ent.text, ent.label_) for ent in doc.ents]
print("\nNamed Entities (first 30):")
for text, label in entities[:30]:
    print(f"{text:30} -> {label}")

(the supreme court, fed, america, this holiday season, the world s richest, saudi, decades, nasa, two decades, the supreme court, fed, america, putin, moscow, hungary, friday, sears, friday, every two minutes, factset research systems inc, chicago, chicago mercantile exchange inc, s p dow jones indices llc, cnn standard, copp clark limited cable, warner bros discovery, cnn, sans cable)

Named Entities (first 30):
the supreme court              -> ORG
fed                            -> ORG
america                        -> GPE
this holiday season            -> DATE
the world s richest            -> ORG
saudi                          -> NORP
decades                        -> DATE
nasa                           -> PERSON
two decades                    -> DATE
the supreme court              -> ORG
fed                            -> ORG
america                        -> GPE
putin                          -> PERSON
moscow                         -> GPE
hungary                        -> GPE
fri

In [8]:
# 7
#Extract all NOUN tokens and PERSON entities
nouns = [
    token.lemma_.lower()
    for token in doc
    if token.pos_ == "NOUN"
]

persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

print("number of NOUN tokens:", len(nouns))
print("number of PERSON entities:", len(persons))
print("\nSample NOUNs:", nouns[:15])
print("Sample PERSONs:", persons[:10])

number of NOUN tokens: 135
number of PERSON entities: 3

Sample NOUNs: ['company', 'thinking', 'retailer', 'holiday', 'season', 'company', 'google', 'analyst', 'investor', 'bull', 'case', 'ai', 'lawsuit', 'medium', 'giant']
Sample PERSONs: ['nasa', 'putin', 'copp clark limited cable']


In [11]:
# 8 - Count Top Nouns & Persons

from collections import Counter
# Count frequencies
noun_counts = Counter(nouns)
person_counts = Counter(persons)

top_nouns = noun_counts.most_common(20)
top_persons = person_counts.most_common(20)

print("🧾 Top 20 Frequent NOUNs")
print("-" * 40)
for word, cnt in top_nouns:
    print(f"{word:<20} {cnt}")

print("\n👩 Top 20 Frequent PERSONs")
print("-" * 40)
for name, cnt in top_persons:
    print(f"{name:<30} {cnt}")


🧾 Top 20 Frequent NOUNs
----------------------------------------
company              5
market               5
holiday              4
season               3
research             3
stock                3
index                3
news                 3
right                3
thinking             2
retailer             2
effort               2
world                2
man                  2
sex                  2
advice               2
decade               2
use                  2
datum                2
time                 2

👩 Top 20 Frequent PERSONs
----------------------------------------
nasa                           1
putin                          1
copp clark limited cable       1
